# InsightForge AI — Corporate Insights Agent
Submission Notebook

“Forging insights from raw data with intelligent planning.”

**Author: Muhammad Ikram Hafeez**

**Track: Enterprise Agents**

In [1]:
# Install once when running from the notebooks directory:
# %pip install -e ../backend

import json
import os

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Markdown, display

from insightforge.core import (
    ConversationMemory,
    DataCatalog,
    InsightForgeAgent,
    PlotArtifact,
    TableArtifact,
)
from insightforge.core.llm import build_llm, offline_fake_llm
from insightforge.ingest import load_any

In [2]:
def generate_synthetic_sales(n_days=180, n_customers=600):
    rng = np.random.default_rng(42)
    dates = pd.date_range(end=pd.Timestamp.today(), periods=n_days)

    products = ["Alpha", "Beta", "Gamma", "Delta"]
    regions = ["North", "South", "East", "West"]
    channels = ["Online", "Retail", "Partner"]

    rows = []
    for date in dates:
        for _ in range(rng.integers(10, 30)):
            product = rng.choice(products)
            region = rng.choice(regions)
            channel = rng.choice(channels)
            units = int(abs(rng.normal(5, 2)) + 1)
            price = float(20 + rng.normal(0, 5))
            revenue = round(units * price, 2)
            customer_id = f"C{rng.integers(1, n_customers):05d}"
            rows.append(
                [date.date(), product, region, channel, units, price, revenue, customer_id]
            )

    return pd.DataFrame(
        rows,
        columns=[
            "date",
            "product",
            "region",
            "channel",
            "units",
            "price",
            "revenue",
            "customer_id",
        ],
    )


sales_df = generate_synthetic_sales()
catalog = DataCatalog()
catalog.register_df("sales", sales_df)
sales_df.head()

,date,product,region,channel,units,price,revenue,customer_id
0,2026-03-27,Delta,East,Retail,7,24.702824,172.92,C00121
1,2026-03-27,Alpha,East,Partner,6,18.418787,110.51,C00308
2,2026-03-27,Alpha,West,Retail,7,23.888960,167.22,C00469
3,2026-03-27,Gamma,South,Partner,6,15.703538,94.22,C00056
4,2026-03-27,Gamma,West,Online,7,19.750370,138.25,C00099


In [3]:
print(catalog.introspect().to_prompt())

TABLE sales (3592 rows)
- date: DATE  e.g. 2026-04-05, 2026-04-08, 2026-04-16
- product: VARCHAR  e.g. Beta, Gamma, Delta
- region: VARCHAR  e.g. North, East, West
- channel: VARCHAR  e.g. Partner, Online, Retail
- units: BIGINT  e.g. 2, 6, 12
- price: DOUBLE  e.g. 21.16084944813128, 7.1667077953435125, 14.735507446480256
- revenue: DOUBLE  e.g. 145.75, 78.92, 156.18
- customer_id: VARCHAR  e.g. C00106, C00562, C00064


In [4]:
has_api_key = bool(os.getenv("LLM_API_KEY") or os.getenv("OPENAI_API_KEY"))
llm = build_llm() if has_api_key else offline_fake_llm()
memory = ConversationMemory()
print(f"Using {'configured OpenAI-compatible' if has_api_key else 'offline fake'} LLM")

Using offline fake LLM


In [5]:
goal = "Show revenue trends over time and identify growth drivers."
result = InsightForgeAgent(llm).run(goal, catalog, memory)

print(json.dumps(result.plan.model_dump(mode="json"), indent=2))
for artifact in result.artifacts:
    if isinstance(artifact, TableArtifact):
        display(Markdown(f"### {artifact.name}"))
        display(pd.DataFrame(artifact.rows))
    elif isinstance(artifact, PlotArtifact):
        pio.show(go.Figure(artifact.figure))

display(Markdown(result.summary))

{
  "steps": [
    {
      "name": "row_count",
      "action": "sql",
      "query": "SELECT COUNT(*) AS row_count FROM \"sales\"",
      "description": ""
    },
    {
      "name": "top_product",
      "action": "sql",
      "query": "SELECT \"product\", COUNT(*) AS count FROM \"sales\" GROUP BY \"product\" ORDER BY count DESC LIMIT 10",
      "description": ""
    },
    {
      "name": "plot_top_product",
      "action": "plot",
      "kind": "bar",
      "data_source": "top_product",
      "x": "product",
      "y": "count",
      "color": null,
      "title": "Top product"
    },
    {
      "name": "stats_units",
      "action": "sql",
      "query": "SELECT MIN(\"units\") AS min, AVG(\"units\") AS avg, MAX(\"units\") AS max FROM \"sales\"",
      "description": ""
    },
    {
      "name": "summary",
      "action": "summary",
      "focus": ""
    }
  ]
}


### row_count

,row_count
0,3592


### top_product

,product,count
0,Delta,920
1,Gamma,902
2,Alpha,894
3,Beta,876


### stats_units

,min,avg,max
0,1,5.500835,14


## Summary

Offline analysis completed for row_count, top_product, stats_units.

In [6]:
follow_up = InsightForgeAgent(llm).run(
    "Which segment should the business investigate next?",
    catalog,
    memory,
)
print(json.dumps(follow_up.plan.model_dump(mode="json"), indent=2))
display(Markdown(follow_up.summary))
print(f"Conversation turns retained: {len(memory.turns)}")

{
  "steps": [
    {
      "name": "row_count",
      "action": "sql",
      "query": "SELECT COUNT(*) AS row_count FROM \"sales\"",
      "description": ""
    },
    {
      "name": "top_product",
      "action": "sql",
      "query": "SELECT \"product\", COUNT(*) AS count FROM \"sales\" GROUP BY \"product\" ORDER BY count DESC LIMIT 10",
      "description": ""
    },
    {
      "name": "plot_top_product",
      "action": "plot",
      "kind": "bar",
      "data_source": "top_product",
      "x": "product",
      "y": "count",
      "color": null,
      "title": "Top product"
    },
    {
      "name": "stats_units",
      "action": "sql",
      "query": "SELECT MIN(\"units\") AS min, AVG(\"units\") AS avg, MAX(\"units\") AS max FROM \"sales\"",
      "description": ""
    },
    {
      "name": "summary",
      "action": "summary",
      "focus": ""
    }
  ]
}


## Summary

Offline analysis completed for row_count, top_product, stats_units.

Conversation turns retained: 2


In [7]:
catalog2 = DataCatalog()
load_any("../backend/tests/fixtures/hr.csv", catalog2)
hr_memory = ConversationMemory()
hr_result = InsightForgeAgent(llm).run(
    "Compare departments and highlight workforce patterns.",
    catalog2,
    hr_memory,
)

print(catalog2.introspect().to_prompt())
print(json.dumps(hr_result.plan.model_dump(mode="json"), indent=2))
for artifact in hr_result.artifacts:
    if isinstance(artifact, TableArtifact):
        display(pd.DataFrame(artifact.rows))
display(Markdown(hr_result.summary))

catalog.close()
catalog2.close()

TABLE hr (60 rows)
- employee_id: BIGINT  e.g. 15, 20, 52
- department: VARCHAR  e.g. Human Resources, Sales, Engineering
- job_title: VARCHAR  e.g. Accountant, Sales Manager, HR Specialist
- hire_date: DATE  e.g. 2019-01-15, 2020-08-14, 2020-02-17
- salary: BIGINT  e.g. 72000, 86000, 127000
- location: VARCHAR  e.g. Remote, Austin, Chicago
- performance_score: DOUBLE  e.g. 3.8, 4.9, 3.7


{
  "steps": [
    {
      "name": "row_count",
      "action": "sql",
      "query": "SELECT COUNT(*) AS row_count FROM \"hr\"",
      "description": ""
    },
    {
      "name": "top_department",
      "action": "sql",
      "query": "SELECT \"department\", COUNT(*) AS count FROM \"hr\" GROUP BY \"department\" ORDER BY count DESC LIMIT 10",
      "description": ""
    },
    {
      "name": "plot_top_department",
      "action": "plot",
      "kind": "bar",
      "data_source": "top_department",
      "x": "department",
      "y": "count",
      "color": null,
      "title": "Top department"
    },
    {
      "name": "stats_employee_id",
      "action": "sql",
      "query": "SELECT MIN(\"employee_id\") AS min, AVG(\"employee_id\") AS avg, MAX(\"employee_id\") AS max FROM \"hr\"",
      "description": ""
    },
    {
      "name": "summary",
      "action": "summary",
      "focus": ""
    }
  ]
}


,row_count
0,60


,department,count
0,Finance,12
1,Sales,12
2,Engineering,12
3,Human Resources,12
4,Marketing,12


,min,avg,max
0,1,30.5,60


## Summary

Offline analysis completed for row_count, top_department, stats_employee_id.